# IBEX 35: seleccion aleatoria de variables con validacion cruzada

El objetivo es predecir `mapfre_close`, entendido como rendimiento porcentual simple del cierre ajustado de Mapfre, usando variables retardadas del IBEX, incluyendo Mapfre en dias anteriores.


In [ ]:
try:
    import yfinance as yf
except ModuleNotFoundError:
    %pip install -q yfinance
    import yfinance as yf

import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import RepeatedKFold, cross_validate
from sklearn.metrics import mean_squared_error, make_scorer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


## Valores del IBEX

La lista usa tickers de Yahoo Finance para los componentes del IBEX 35. Los nombres de la izquierda son los prefijos que tendran las columnas del dataframe final.


In [ ]:
IBEX_TICKERS = {
    "acs": "ACS.MC",
    "acerinox": "ACX.MC",
    "aena": "AENA.MC",
    "amadeus": "AMS.MC",
    "acciona": "ANA.MC",
    "accionaenergia": "ANE.MC",
    "bbva": "BBVA.MC",
    "bankinter": "BKT.MC",
    "caixabank": "CABK.MC",
    "cellnex": "CLNX.MC",
    "colonial": "COL.MC",
    "endesa": "ELE.MC",
    "enagas": "ENG.MC",
    "fluidra": "FDR.MC",
    "ferrovial": "FER.MC",
    "grifols": "GRF.MC",
    "iag": "IAG.MC",
    "iberdrola": "IBE.MC",
    "indra": "IDR.MC",
    "inditex": "ITX.MC",
    "logista": "LOG.MC",
    "mapfre": "MAP.MC",
    "merlin": "MRL.MC",
    "arcelormittal": "MTS.MC",
    "naturgy": "NTGY.MC",
    "puig": "PUIG.MC",
    "redeia": "RED.MC",
    "repsol": "REP.MC",
    "rovi": "ROVI.MC",
    "sabadell": "SAB.MC",
    "santander": "SAN.MC",
    "sacyr": "SCYR.MC",
    "solaria": "SLR.MC",
    "telefonica": "TEF.MC",
    "unicaja": "UNI.MC",
}


## Descarga y preparacion inicial

La funcion `descarga` baja el cierre ajustado y el volumen. Las columnas `*_close` son rendimientos porcentuales simples calculados como `100 * (precio_t / precio_t-1 - 1)`; las columnas `*_vol` conservan el volumen diario.


In [ ]:
def extrae_serie(datos, ticker, campo):
    """Devuelve una serie de yfinance tanto si el MultiIndex viene por ticker como por campo."""
    if not isinstance(datos.columns, pd.MultiIndex):
        return datos[campo]

    nivel_0 = datos.columns.get_level_values(0)
    nivel_1 = datos.columns.get_level_values(1)

    if ticker in nivel_0 and campo in nivel_1:
        return datos[(ticker, campo)]
    if campo in nivel_0 and ticker in nivel_1:
        return datos[(campo, ticker)]

    raise KeyError(f"No encuentro {campo} para {ticker}")


def descarga(fecha_inicial="2021-01-01", tickers=IBEX_TICKERS):
    """Descarga datos de Yahoo Finance y devuelve close pct-ret y volumen del IBEX."""
    lista_yahoo = list(tickers.values())
    datos = yf.download(
        lista_yahoo,
        start=fecha_inicial,
        auto_adjust=False,
        group_by="ticker",
        progress=False,
        threads=True,
    )

    columnas = {}
    avisos = []

    for nombre, ticker in tickers.items():
        try:
            cierre_ajustado = extrae_serie(datos, ticker, "Adj Close")
            volumen = extrae_serie(datos, ticker, "Volume")
        except KeyError as exc:
            avisos.append(str(exc))
            continue

        columnas[f"{nombre}_close"] = cierre_ajustado.pct_change() * 100
        columnas[f"{nombre}_vol"] = volumen

    df = pd.DataFrame(columnas).replace([np.inf, -np.inf], np.nan)
    columnas_vacias = df.columns[df.isna().all()].tolist()
    if columnas_vacias:
        print("Columnas sin datos en Yahoo Finance y eliminadas:")
        print(columnas_vacias)
        df = df.drop(columns=columnas_vacias)

    if avisos:
        print("Avisos de descarga:")
        for aviso in avisos:
            print("-", aviso)

    return df


In [ ]:
df = descarga("2021-01-01")
df.head()


In [ ]:
df.shape


## Construccion de `df_p`

En cada fila dejamos el objetivo `c` del dia actual y, como variables explicativas, todas las columnas retardadas de 1 a `p` dias, incluida `mapfre_close` de dias anteriores. No se usa `mapfre_close` del propio dia como predictor.


In [ ]:
def prepara_df_p(df, c="mapfre_close", p=5):
    if c not in df.columns:
        raise ValueError(f"La columna objetivo {c!r} no esta en df")

    objetivo = df[[c]].copy()
    explicativas = df.copy()
    retardos = []

    for lag in range(1, p + 1):
        retardos.append(explicativas.shift(lag).add_suffix(f"_t-{lag}"))

    df_p = pd.concat([objetivo] + retardos, axis=1).dropna()
    return df_p


In [ ]:
c = "mapfre_close"
p = 5

df_p = prepara_df_p(df, c=c, p=p)

assert c in df_p.columns
assert any(col.startswith(f"{c}_t-") for col in df_p.columns), "Faltan retardos de Mapfre"

df_p.head()


In [ ]:
for c in df_p.columns:
    print(c,end=" ")

In [ ]:
df_p.shape


## Validacion cruzada

Se sigue el esquema de `19cv.ipynb`: `LinearRegression`, `RepeatedKFold`, `cross_validate`, `mean_squared_error` negativo y conversion posterior a RMSE. Como mezclamos rendimientos y volumenes, la regresion se evalua dentro de un `Pipeline` con `StandardScaler` para escalar las variables predictoras en cada fold.


In [ ]:
def LinearRegressionCV(X, y, repeticiones=100, divisiones=10, random_state=None):
    medidor_error = make_scorer(mean_squared_error, greater_is_better=False)
    repartidor = RepeatedKFold(
        n_splits=divisiones,
        n_repeats=repeticiones,
        random_state=random_state,
    )
    metodo = make_pipeline(StandardScaler(), LinearRegression())
    cv_res = cross_validate(
        metodo,
        X,
        y,
        cv=repartidor,
        scoring=medidor_error,
        n_jobs=-1,
    )

    mse_neg = cv_res["test_score"]
    rmse = np.sqrt(-mse_neg)
    return rmse.mean()


## Busqueda aleatoria de columnas

En cada iteracion se eligen `m` columnas al azar entre las variables retardadas y se calcula el RMSE medio con validacion cruzada.


In [ ]:
n = 200
m = 12
repeticiones = 100
divisiones = 10
semilla = None

XColumns = [col for col in df_p.columns if col != c]
y = df_p[c]

rng = np.random.default_rng(semilla)
resultados = []

for i in range(n):
    columnas_elegidas = list(rng.choice(XColumns, size=m, replace=False))
    X = df_p[columnas_elegidas]
    rmse_medio = LinearRegressionCV(
        X,
        y,
        repeticiones=repeticiones,
        divisiones=divisiones,
        #random_state=semilla,
    )
    resultados.append({
        "iteracion": i + 1,
        "columnas": columnas_elegidas,
        "rmse": rmse_medio,
    })

    if (i + 1) % 10 == 0:
        print(f"Iteracion {i + 1}/{n}. Mejor RMSE provisional: {min(r['rmse'] for r in resultados):.6f}")

df_res = pd.DataFrame(resultados).sort_values("rmse").reset_index(drop=True)
df_res


## Estadisticas de los experimentos

Calculamos un pequeno resumen de los `n` experimentos, incluyendo el porcentaje de mejora del mejor RMSE frente al peor.


In [ ]:
rmse_mejor = df_res["rmse"].min()
rmse_peor = df_res["rmse"].max()
mejora_pct = 100 * (rmse_peor - rmse_mejor) / rmse_peor

estadisticas_rmse = pd.DataFrame({
    "estadistico": ["mejor", "peor", "media", "mediana", "desviacion_tipica", "mejora_mejor_vs_peor_%"],
    "valor": [
        rmse_mejor,
        rmse_peor,
        df_res["rmse"].mean(),
        df_res["rmse"].median(),
        df_res["rmse"].std(),
        mejora_pct,
    ],
})

print(f"Mejora del mejor RMSE frente al peor: {mejora_pct:.2f}%")
estadisticas_rmse


## Mejor combinacion encontrada

In [ ]:
mejor = df_res.iloc[0]
mejores_columnas = mejor["columnas"]

print(f"Menor RMSE medio: {mejor['rmse']:.6f}")
print("Columnas seleccionadas:")
for columna in mejores_columnas:
    print("-", columna)


In [ ]:
mejores_columnas
